In [8]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../cms_hospital.db")
paradox_hospitals = pd.read_csv('../outputs/paradox_hospitals.csv')
paradox_hospitals['facility_id'] = paradox_hospitals['facility_id'].astype(str).str.zfill(6)


#### Layer 3 — Identify the process failures

*Do high-spend hospitals have worse infection rates?*

*Higher readmissions?*

*More unplanned visits?*

*Where is the money actually going?*

In Layer 2 we established that the 99 paradox hospitals are:
- All Acute Care facilities — fully resourced, no excuse of being small or rural
- Spread across all ownership types — not a for-profit problem
- Disproportionately concentrated in smaller southern and midwestern states

We know **who** they are. Now we investigate **why** they underperform.

The hypothesis: these hospitals spend more because their internal processes
are failing — generating avoidable costs through infections, readmissions,
and unplanned visits. The spending is not an investment in quality.
It is the financial consequence of dysfunction.

**Three process dimensions to investigate:**
1. **Infections (HAI)** — Are SIR scores worse than the national benchmark?
2. **Readmissions** — Are patients coming back after discharge?
3. **Unplanned Visits** — Are patients returning to the ER unexpectedly?

In [4]:
hai_comparison = pd.read_sql_query("""
    WITH paradox_hai AS (
        SELECT
            'Paradox Hospitals' AS group_name,
            ROUND(AVG(HAI_1_SIR), 3) AS avg_HAI_1,
            ROUND(AVG(HAI_2_SIR), 3) AS avg_HAI_2,
            ROUND(AVG(HAI_3_SIR), 3) AS avg_HAI_3,
            ROUND(AVG(HAI_4_SIR), 3) AS avg_HAI_4,
            ROUND(AVG(HAI_5_SIR), 3) AS avg_HAI_5,
            ROUND(AVG(HAI_6_SIR), 3) AS avg_HAI_6
        FROM healthcare_infections
        WHERE facility_id IN ({})
    ),
    non_paradox_hai AS (
        SELECT
            'All Other Hospitals' AS group_name,
            ROUND(AVG(HAI_1_SIR), 3) AS avg_HAI_1,
            ROUND(AVG(HAI_2_SIR), 3) AS avg_HAI_2,
            ROUND(AVG(HAI_3_SIR), 3) AS avg_HAI_3,
            ROUND(AVG(HAI_4_SIR), 3) AS avg_HAI_4,
            ROUND(AVG(HAI_5_SIR), 3) AS avg_HAI_5,
            ROUND(AVG(HAI_6_SIR), 3) AS avg_HAI_6
        FROM healthcare_infections
        WHERE facility_id NOT IN ({})
    )
    SELECT * FROM paradox_hai
    UNION ALL
    SELECT * FROM non_paradox_hai
""".format(
    ','.join(['?']*len(paradox_hospitals)),
    ','.join(['?']*len(paradox_hospitals))
), conn, params=paradox_hospitals['facility_id'].tolist()*2)

print("SIR = 1.0 is the national benchmark.")
hai_comparison

SIR = 1.0 is the national benchmark.


,group_name,avg_HAI_1,avg_HAI_2,avg_HAI_3,avg_HAI_4,avg_HAI_5,avg_HAI_6
0,Paradox Hospitals,1.074,0.841,1.386,1.581,1.143,0.518
1,All Other Hospitals,0.590,0.506,0.831,1.072,0.677,0.391


## Finding - Infection Rates: Paradox Hospitals Are Significantly Worse

Comparing average SIR (Standardized Infection Ratio) scores between
the 99 paradox hospitals and all other hospitals reveals a consistent pattern.
SIR = 1.0 is the national benchmark. Above 1.0 = worse than expected.

| Measure | Paradox Hospitals | All Other Hospitals |
|---------|------------------|---------------------|
| HAI_1 (CLABSI) | 1.074 | 0.590 |
| HAI_2 (CAUTI) | 0.841 | 0.506 |
| HAI_3 (SSI Colon) | 1.386 | 0.831 |
| HAI_4 (MRSA) | 1.581 | 1.072 |
| HAI_5 (C.diff) | 1.143 | 0.677 |
| HAI_6 (SSI Abdominal) | 0.518 | 0.391 |

**Key insight:** All other hospitals perform **below 1.0** on most measures —
meaning they are actually better than the national average.
Paradox hospitals are **above 1.0 on 5 out of 6 measures** —
consistently worse than the national benchmark.

The most alarming gaps are in HAI_4 (MRSA) and HAI_3 (SSI Colon) —
both nearly double the rate of all other hospitals.

This is the first concrete process failure identified in the paradox group.
These hospitals are not just spending more —
they are generating avoidable infections at a significantly higher rate.
Every infection extends a patient's stay, requires additional treatment,
and drives up the cost of care.
The spending is not investment. It is consequence.

In [11]:
readmissions_comparison = pd.read_sql_query("""
    WITH paradox_readm AS (
        SELECT
            'Paradox Hospitals' AS group_name,
            ROUND(AVG(READM_30_AMI), 3)      AS avg_READM_30_AMI,
            ROUND(AVG(READM_30_CABG), 3)     AS avg_READM_30_CABG,
            ROUND(AVG(READM_30_COPD), 3)     AS avg_READM_30_COPD,
            ROUND(AVG(READM_30_HF), 3)       AS avg_READM_30_HF,
            ROUND(AVG(READM_30_HIP_KNEE), 3) AS avg_READM_30_HIP_KNEE,
            ROUND(AVG(READM_30_PN), 3)       AS avg_READM_30_PN
        FROM unplanned_visits
        WHERE facility_id IN ({})
    ),
    non_paradox_readm AS (
        SELECT
            'All Other Hospitals' AS group_name,
            ROUND(AVG(READM_30_AMI), 3)      AS avg_READM_30_AMI,
            ROUND(AVG(READM_30_CABG), 3)     AS avg_READM_30_CABG,
            ROUND(AVG(READM_30_COPD), 3)     AS avg_READM_30_COPD,
            ROUND(AVG(READM_30_HF), 3)       AS avg_READM_30_HF,
            ROUND(AVG(READM_30_HIP_KNEE), 3) AS avg_READM_30_HIP_KNEE,
            ROUND(AVG(READM_30_PN), 3)       AS avg_READM_30_PN
        FROM unplanned_visits
        WHERE facility_id NOT IN ({})
    )
    SELECT * FROM paradox_readm
    UNION ALL
    SELECT * FROM non_paradox_readm
""".format(
    ','.join(['?']*len(paradox_hospitals)),
    ','.join(['?']*len(paradox_hospitals))
), conn, params=paradox_hospitals['facility_id'].tolist()*2)
print("The national context: CMS considers anything above ~20% for HF as a red flag")
readmissions_comparison

The national context: CMS considers anything above ~20% for HF as a red flag


,group_name,avg_READM_30_AMI,avg_READM_30_CABG,avg_READM_30_COPD,avg_READM_30_HF,avg_READM_30_HIP_KNEE,avg_READM_30_PN
0,Paradox Hospitals,13.752,10.602,18.289,19.807,4.734,16.297
1,All Other Hospitals,13.627,10.638,18.227,19.705,4.860,15.984


#### Finding — Readmission Rates: Marginal Difference

Comparing 30-day readmission rates between the 99 paradox hospitals
and all other hospitals shows only small differences.

| Measure | Paradox Hospitals | All Other Hospitals | Difference |
|---------|------------------|---------------------|------------|
| READM_30_AMI | 13.752% | 13.627% | +0.125% |
| READM_30_CABG | 10.602% | 10.638% | -0.036% |
| READM_30_COPD | 18.289% | 18.227% | +0.062% |
| READM_30_HF | 19.807% | 19.705% | +0.102% |
| READM_30_HIP_KNEE | 4.734% | 4.860% | -0.126% |
| READM_30_PN | 16.297% | 15.984% | +0.313% |

**Key insight:** Unlike infections, readmission rates show no meaningful
gap between paradox hospitals and all others.
The differences are fractions of a percentage point —
too small to be considered a primary driver of the paradox.

Readmissions are not where these hospitals are failing.
The process failure identified in infection rates is a much stronger signal.

In [10]:
unplanned_comparison = pd.read_sql_query("""
    WITH paradox_unplanned AS (
        SELECT
            'Paradox Hospitals' AS group_name,
            ROUND(AVG(EDAC_30_AMI), 3) AS avg_EDAC_30_AMI,
            ROUND(AVG(EDAC_30_HF), 3)  AS avg_EDAC_30_HF,
            ROUND(AVG(EDAC_30_PN), 3)  AS avg_EDAC_30_PN,
            ROUND(AVG(OP_32), 3)       AS avg_OP_32,
            ROUND(AVG(OP_35_ADM), 3)   AS avg_OP_35_ADM,
            ROUND(AVG(OP_35_ED), 3)    AS avg_OP_35_ED,
            ROUND(AVG(OP_36), 3)       AS avg_OP_36
        FROM unplanned_visits
        WHERE facility_id IN ({})
    ),
    non_paradox_unplanned AS (
        SELECT
            'All Other Hospitals' AS group_name,
            ROUND(AVG(EDAC_30_AMI), 3) AS avg_EDAC_30_AMI,
            ROUND(AVG(EDAC_30_HF), 3)  AS avg_EDAC_30_HF,
            ROUND(AVG(EDAC_30_PN), 3)  AS avg_EDAC_30_PN,
            ROUND(AVG(OP_32), 3)       AS avg_OP_32,
            ROUND(AVG(OP_35_ADM), 3)   AS avg_OP_35_ADM,
            ROUND(AVG(OP_35_ED), 3)    AS avg_OP_35_ED,
            ROUND(AVG(OP_36), 3)       AS avg_OP_36
        FROM unplanned_visits
        WHERE facility_id NOT IN ({})
    )
    SELECT * FROM paradox_unplanned
    UNION ALL
    SELECT * FROM non_paradox_unplanned
""".format(
    ','.join(['?']*len(paradox_hospitals)),
    ','.join(['?']*len(paradox_hospitals))
), conn, params=paradox_hospitals['facility_id'].tolist()*2)

unplanned_comparison

,group_name,avg_EDAC_30_AMI,avg_EDAC_30_HF,avg_EDAC_30_PN,avg_OP_32,avg_OP_35_ADM,avg_OP_35_ED,avg_OP_36
0,Paradox Hospitals,13.470,12.893,17.988,12.826,11.081,5.385,0.971
1,All Other Hospitals,6.841,4.918,5.871,13.001,10.823,5.400,1.007


#### Finding — Unplanned Visits: The Strongest Signal Yet
- EDAC = patients staying longer than expected after discharge
- OP = patients returning unexpectedly after being sent home

Comparing excess acute care days (EDAC) and unplanned return visits (OP)
between the 99 paradox hospitals and all other hospitals reveals
the most alarming gap found so far.

| Measure | Paradox Hospitals | All Other Hospitals | Difference |
|---------|------------------|---------------------|------------|
| EDAC_30_AMI | 13.470 | 6.841 | +97% worse |
| EDAC_30_HF | 12.893 | 4.918 | +162% worse |
| EDAC_30_PN | 17.988 | 5.871 | +206% worse |
| OP_32 | 12.826 | 13.001 | -1.3% better |
| OP_35_ADM | 11.081 | 10.823 | +2.4% worse |
| OP_35_ED | 5.385 | 5.400 | negligible |
| OP_36 | 0.971 | 1.007 | negligible |

**Key insight:** The EDAC measures tell a devastating story.
Paradox hospitals generate 2x to 3x more excess acute care days
than all other hospitals — meaning patients are staying far longer
than clinically expected after heart attacks, heart failure, and pneumonia.

The OP measures show no meaningful difference —
patients are not returning to the ER at higher rates.
The failure is happening **inside the hospital, not after discharge.**


## Layer 3 — Summary: The Mechanism of the Paradox

Three process dimensions were investigated. Here is what the data shows:

| Process Dimension | Signal Strength | Finding |
|-------------------|----------------|---------|
| Infections (HAI) | 🔴 Strong | SIR scores 2x worse than peers |
| Readmissions | 🟡 Weak | Marginal difference, not a primary driver |
| Unplanned Visits / Excess Days | 🔴 Very Strong | EDAC rates 2-3x worse than peers |

**The mechanism is now visible:**

These hospitals are keeping patients longer than necessary.
Extended stays increase infection exposure.
Infections extend stays further.
Each failure feeds the next — and all of it drives up spending.

The spending is not the cause of poor outcomes.
It is the financial consequence of two compounding process failures:
**excess acute care days and avoidable infections.**

This is why money alone cannot fix these hospitals.
You cannot spend your way out of a process problem.

#### The Core Mechanism — Why These Hospitals Keep Patients Longer

The data confirms that paradox hospitals generate 2-3x more excess acute
care days than their peers — and CMS already risk-adjusts EDAC for patient
severity. This means the extended stays cannot be explained by sicker patients.
The hospital itself is the variable.

**Why do these hospitals keep patients longer?**

The data points to a compounding cycle of process failures:

**1 — Infections created inside the hospital**
HAI rates are 2x worse than peers. An infection discovered mid-stay
turns a 4-day admission into a 10-day one. The hospital generated
the complication that extended the stay.

**2 — Discharge planning failures**
Patients may be medically stable but have nowhere safe to go —
no coordination with post-acute care, rehab, or home health services.
The patient stays not because they need acute care,
but because the transition out was never organized.

**3 — Care coordination gaps**
Different departments not communicating — patients waiting for
test results, specialist consultations, or procedures that fall
through the cracks of a fragmented internal system.

**4 — Understaffing or skill gaps**
Without the right staff to stabilize patients quickly,
recovery timelines extend. The intervention that should happen
on day 2 happens on day 5.

**5 — Financial incentives** *(uncomfortable but documented)*
In some hospital structures, longer stays generate more billing
opportunities — per-day reimbursements, additional procedures,
extended monitoring. The system does not always reward efficiency.


#### What the Data Can and Cannot Tell Us

This analysis confirms the **what:**
- Longer stays than expected ✅
- Higher infection rates ✅
- Worse clinical outcomes ✅
- Higher spending ✅

The **why** behind the extended stays — staffing levels,
discharge planning quality, internal coordination — requires
data this dataset does not contain.

That is not a weakness of this analysis.
That is the finding itself:

> **The problem is operational, not clinical.**
> These hospitals are not treating sicker patients.
> They are managing their processes worse —
> and patients are paying for it with their health,
> and the system is paying for it with its money.

The final question remaining is: **where exactly is that money going?**
That is what the spending breakdown will reveal.

In [14]:
spending_breakdown = pd.read_sql_query("""
    WITH paradox_spend AS (
        SELECT
            'Paradox Hospitals' AS group_name,
            ROUND(AVG(complete_total), 0)      AS avg_total,
            ROUND(AVG(during_inpatient), 0)    AS avg_during_inpatient,
            ROUND(AVG(during_outpatient), 0)   AS avg_during_outpatient,
            ROUND(AVG(during_carrier), 0)      AS avg_during_carrier,
            ROUND(AVG(after_inpatient), 0)     AS avg_after_inpatient,
            ROUND(AVG(after_snf), 0)           AS avg_after_snf,
            ROUND(AVG(after_home_health), 0)   AS avg_after_home_health,
            ROUND(AVG(prior_inpatient), 0)     AS avg_prior_inpatient
        FROM medicare_spending
        WHERE facility_id IN ({})
    ),
    non_paradox_spend AS (
        SELECT
            'All Other Hospitals' AS group_name,
            ROUND(AVG(complete_total), 0)      AS avg_total,
            ROUND(AVG(during_inpatient), 0)    AS avg_during_inpatient,
            ROUND(AVG(during_outpatient), 0)   AS avg_during_outpatient,
            ROUND(AVG(during_carrier), 0)      AS avg_during_carrier,
            ROUND(AVG(after_inpatient), 0)     AS avg_after_inpatient,
            ROUND(AVG(after_snf), 0)           AS avg_after_snf,
            ROUND(AVG(after_home_health), 0)   AS avg_after_home_health,
            ROUND(AVG(prior_inpatient), 0)     AS avg_prior_inpatient
        FROM medicare_spending
        WHERE facility_id NOT IN ({})
    )
    SELECT * FROM paradox_spend
    UNION ALL
    SELECT * FROM non_paradox_spend
""".format(
    ','.join(['?']*len(paradox_hospitals)),
    ','.join(['?']*len(paradox_hospitals))
), conn, params=paradox_hospitals['facility_id'].tolist()*2)

print("Each number shown covers everything Medicare pays for that patient across that entire window - the hospital stay, the doctors, any skilled nursing after, home health after discharge")
print("NOTE: An episode is : 3 days before admission - The entire hospital stay (however long it is) - 30 days after discharge")
spending_breakdown

Each number shown covers everything Medicare pays for that patient across that entire window - the hospital stay, the doctors, any skilled nursing after, home health after discharge
NOTE: An episode is : 3 days before admission - The entire hospital stay (however long it is) - 30 days after discharge


,group_name,avg_total,avg_during_inpatient,avg_during_outpatient,avg_during_carrier,avg_after_inpatient,avg_after_snf,avg_after_home_health,avg_prior_inpatient
0,Paradox Hospitals,29059.0,12297.0,0.0,1513.0,4932.0,5330.0,689.0,45.0
1,All Other Hospitals,25574.0,11488.0,0.0,1287.0,3866.0,4356.0,716.0,39.0


## Where Is the Money Actually Going?

The $3,485 gap per patient episode between paradox hospitals and all others
does not come from one place — it is spread across the entire care journey.

| Spending Category | Paradox Hospitals | All Other Hospitals | Gap |
|-------------------|------------------|---------------------|-----|
| During — Inpatient | 12,297 | 11,488 | +809 |
| During — Carrier | 1,513 | 1,287 | +226 |
| After — Inpatient | 4,932 | 3,866 | +1,066 |
| After — SNF | 5,330 | 4,356 | +974 |
| After — Home Health | 689 | 716 | -27 |
| Prior — Inpatient | 45 | 39 | +6 |
| **Total** | **29,059** | **25,574** | **+3,485** |


**The biggest gaps are not during the stay — they are after it:**

- **+ 1,066 dollars after inpatient** → patients getting readmitted after discharge
- **+ 974 dollars after SNF** → patients leaving the hospital but not going home —
  they need skilled nursing facility care because they were not fully stabilized
- **+ 809 dollars during inpatient** → longer stays costing more per admission

This is the financial fingerprint of the process failures identified earlier.
The hospital's internal dysfunction — excess stay days, avoidable infections,
poor discharge planning — does not stay inside the hospital.
It follows the patient out the door and keeps generating costs
for 30 days after they leave.

At scale: 99 paradox hospitals × average excess of $3,485 per patient
represents hundreds of millions in excess Medicare spending annually —
with worse clinical outcomes to show for it.

#### Note — Understanding the Spending Categories

CMS measures spending across three periods of a patient episode:

**PRIOR** — 3 days before hospital admission
- `prior_inpatient` → any inpatient costs in the days leading up to admission

**DURING** — the hospital stay itself
- `during_inpatient` → the core hospital bill — room, nursing, procedures
- `during_outpatient` → outpatient services used during the stay
- `during_carrier` → physician fees during the stay (doctors, specialists)

**AFTER** — 30 days following discharge
- `after_inpatient` → patient was readmitted to a hospital after being discharged
- `after_snf` → patient was sent to a Skilled Nursing Facility instead of going home
- `after_home_health` → nurse or therapist visits to the patient at home
- `after_carrier` → physician follow-up visits after discharge
- `after_dme` → medical equipment sent home with the patient (wheelchair, oxygen)
- `after_hospice` → end of life care costs after discharge

**The most telling categories for our analysis:**
- High `after_inpatient` → patients bouncing back into hospitals
- High `after_snf` → patients not stable enough to go straight home
- High `during_inpatient` → patients staying longer than expected

Together these three tell you whether a hospital is
truly finishing the job or just opening the door and hoping for the best.


## The Paradox in One Paragraph

Imagine a patient admitted for heart failure with an expected stay of 4 days.
At a paradox hospital, that patient stays 5, 6, maybe 7 days.
During that extended stay they pick up a hospital-acquired infection.
The infection requires treatment. The stay gets longer.
The longer stay creates more complications.
More complications require more interventions.
The patient gets worse. The costs pile up.

And here is the cruel irony:
The hospital cannot discharge an unstable patient.
That is illegal and unethical.
So the hospital that created the problem through poor processes
is now legally required to keep treating it —
paying for every extra day, every infection treatment,
every specialist called in to fix what should never have happened.

Then the patient is finally discharged — but not fully stabilized.
So within 30 days they end up in a skilled nursing facility
or back in a hospital. More costs. Worse outcomes.

**The hospital that runs the worst processes
ends up spending the most money per patient —
not because they are doing more for the patient,
but because they keep having to fix their own mistakes.**

That is not investment in care.
That is the financial cost of dysfunction.
And that is the paradox this analysis set out to prove.